This is the second part of the [User guide](../../usage.qmd), between [dataset and preprocessing](../../data.qmd) and [installation and use](../00_match_workflow/00_match_workflow.ipynb).

## 1. Three distinct units {#three-distinct-units}

- **Individual possession:** one player’s control interval, identified by `(match_id, possession_id)`. Its observed start and end must be matched to tracking for the interval table used here.
- **Team possession:** a chain of individual intervals connected by verified completed passes. A completed pass changes the carrier without ending this team interval. Unverified continuity starts a new reconstructed team interval.
- **Canonical action:** a one-second carry attempt or an instantaneous pass release. Its start state is a checkpoint at which the hazard h, the probability that the team loses possession during the next action, is evaluated.

A *checkpoint* is the start state of a canonical action. An *additional snapshot* is any other evaluated state; it does not define an action.

Frame evaluation is more general than action segmentation. A usable frame can be evaluated even if it is not a checkpoint. Segmentation supplies the conventional discrete action sequence underlying h; extra evaluations do not create additional independent actions [@fernandez2021].

## 2. Canonical checkpoints {#canonical-checkpoints}

The checkpoints of an individual possession are the starts of its complete one-second carries, plus one terminal checkpoint that depends on how control ends: the release after a pass, the start of the final failed carry after a direct loss, and none after a foul suffered or a clearance.

Carries are anchored to the **start of the individual possession**, in successive one-second blocks. Each full successful carry supplies one checkpoint at its start. If control ends in a direct possession loss, the final failed attempt is retained even when it lasts less than one second. If loss occurs exactly at a full-second boundary, the last full block is the failed attempt.

A pass supplies a checkpoint **at release**, regardless of how much time has elapsed since the last full carry. A successful subsecond tail before a pass does not become another canonical carry. It can be inspected as an additional snapshot, but is excluded from the final one-second carry benchmark and action-choice training.

Fouls suffered and clearances preserve preceding complete carries. They do not supply a terminal pass/carry action; their terminal frames and short tails can be retained as additional snapshots. Shots are outside the fitted pass–carry task. The [composed-risk benchmark](../../benchmarks/06_composed_risk.qmd) separately defines the terminally covered cohort used for bands and reports.

`get_checkpoints()` selects canonical action starts by default. `include_snapshots=True` also returns the annotated extras, marked by `is_canonical=False` and `checkpoint_kind="snapshot"`. `compute_player_possession(..., include_snapshots=False)` evaluates the canonical sequence only; `frame_ids` supports an explicitly chosen sampling. Extra overlapping forecasts must not be multiplied into a canonical survival curve.

### Possessions shorter than one second {#short-possessions}

A short duration does not by itself remove an action. What matters is how control ends:

- **Pass before 1 s:** only the release is a canonical checkpoint. The earlier control interval is shorter than a full carry, so it supplies no successful canonical carry. The start can be inspected as an additional snapshot.
- **Direct loss before 1 s:** retain one failed carry attempt, with its checkpoint at the individual-possession start. It is a failure within the fixed one-second forecast horizon, even though the observed loss happens earlier. The loss endpoint is not another action start.
- **Foul suffered or clearance before 1 s:** no complete carry and no terminal pass/carry action are available, so there is no canonical checkpoint. Annotated snapshots remain available for inspection.

The canonical profile of a short pass or a short failed carry therefore contains one point: the release for a pass, the start of the possession for a failed carry. Adding another evaluation frame does not turn it into a second canonical action.

### A partial tail after full carries {#partial-tails}

For a possession lasting **2.4 s and ending in a pass**, the successful canonical carries remain **0–1 s and 1–2 s**. The last **2–2.4 s** is a short control tail, not a third complete carry. Canonical checkpoints are **0, 1 and 2.4 s**; **2 s** is an optional snapshot.

If the same 2.4 s duration ends in a **direct carry loss**, retain the failed **2–2.4 s** attempt. Canonical checkpoints are then **0, 1 and 2 s**. The terminal action is evaluated at 2 s, before execution fails; there is no extra decision at 2.4 s.

For a **foul suffered or clearance at 2.4 s**, only the carry starts at **0 and 1 s** are canonical. The tail start at 2 s and terminal frame at 2.4 s are additional snapshots. They do not supply a terminal canonical action. These timings illustrate the segmentation rule, not additional match observations or benchmark samples.

## 3. Initial and final states {#initial-final-state}

The initial state of an individual possession is its first canonical checkpoint. The final state is its last canonical checkpoint when control ends with a pass or a direct loss: the release in the first case and the start of the final failed carry in the second. A foul suffered or a clearance supplies no terminal action, so the possession has no final state in this sense. A possession shorter than one second has a single checkpoint, which is both its initial and its final state.

The table applies the checkpoint rules and these definitions to example durations: any duration below 1 s, and 2.0 s and 2.4 s as examples of a duration that is, or is not, a whole number of seconds. Other durations follow the same rules.

| Ending | Example duration | Canonical checkpoints (s) | Initial state | Final state |
|---|---|---|---|---|
| Pass | 2.4 s | 0, 1, 2.4 | 0 s | Release at 2.4 s |
| Pass | 2.0 s | 0, 1, 2.0 | 0 s | Release at 2.0 s |
| Pass | below 1 s | Release only | Release | Release |
| Direct loss | 2.4 s | 0, 1, 2 | 0 s | Carry start at 2 s |
| Direct loss | 2.0 s | 0, 1 | 0 s | Carry start at 1 s |
| Direct loss | below 1 s | 0 | Possession start | Possession start |
| Foul suffered or clearance | 2.4 s | 0, 1 | 0 s | None |
| Foul suffered or clearance | below 1 s | None | None | None |

The pressure patterns of the [case studies](../07_case_studies/07_case_studies.ipynb) and the band cohort of the [composed-risk benchmark](../../benchmarks/06_composed_risk.qmd) use these definitions.

## 4. Start type and previous action {#previous-action}

The pass-selection, action-selection and residual-risk models use two possession-level context variables, `start_type` and `previous_action`. The [table of context variables by task](#context-in-tasks) lists what each task reads. This section fixes the convention for both.

**Start type** describes how the player obtained control. It is taken without modification from the Dynamic Events of SkillCorner, which define the categories and their annotation [@skillcorner-opendata]. The labels are listed [below](#context-labels).

**Previous action** follows the convention of [@stalpaert2022]: when an action is preceded by a complete carry, the previous action is the carry, named `ball_drive` in the code. Otherwise the previous action is the start type. With the one-second carries of this segmentation, the rule is applied to the observed prefix of the individual possession:

- Before a complete second of control, `previous_action` equals `start_type`.
- From 1.0 s of control onwards, after at least one complete carry, `previous_action` is `ball_drive`; the checkpoint at 1 s already carries it.
- The same rule applies to a pass release or an arbitrary intermediate frame.

Thus a quick pass after reception keeps the reception context, such as `pass_reception`, while a pass after several seconds of carrying has `previous_action="ball_drive"`. For example, a pass released at 2.4 s follows two complete carries and has `previous_action="ball_drive"`, whereas a pass released before 1 s keeps the start type. The field is not the previous row in the global event table and does not describe how the current action will end.

## 5. Context labels and their origin {#context-labels}

The learned models use the granular `start_type` labels of the SkillCorner Dynamic Events without regrouping them. `previous_action` takes the same labels until a complete carry is observed, and then becomes `ball_drive`. In feature records these columns are named `start_type_raw` and `previous_action_raw`. SkillCorner defines the labels [@skillcorner-opendata]; the table gives a short general reading of each, and the football concepts (reception, interception, recovery and restart context) are not specific to this provider.

| Label | General meaning |
|---|---|
| `pass_reception` | Receiving an open-play pass. |
| `pass_interception` | Obtaining control by intercepting an opposing open-play pass. |
| `recovery` | Obtaining control of a loose or available ball. |
| `keep_possession` | Retaining or re-establishing control, as distinguished from a new reception or interception by the source annotation. |
| `throw_in_reception`, `corner_reception`, `free_kick_reception`, `goal_kick_reception` | Receiving the ball from the named restart. |
| `throw_in_interception`, `corner_interception`, `free_kick_interception`, `goal_kick_interception` | Gaining control from the opponent’s named restart. |
| `unknown` | The source explicitly leaves the start context unspecified. |
| `ball_drive` | Carrying after at least one complete canonical carry; generated by PRESS for `previous_action`, not an individual-possession start label. |

A possession may begin with a restart reception and continue in active play. Keeping that **start context** is distinct from admitting a dead-ball delivery into the open-play pass-execution cohort. Pass-link and outcome filters belong to the individual [task benchmarks](../../benchmark.qmd).

The fitted encoders also contain `__missing__` and `__unseen__` buckets for an absent value or a category outside the fitted vocabulary. They are encoding markers, not football actions. Legacy `_std` columns group the raw labels into open-play/set-piece reception, open-play/set-piece interception and continuation; the deployed choice and residual-risk models use the granular vocabulary instead.

For another provider, map equivalent event meanings to these labels before using the pretrained models. Preserve genuinely unavailable context as missing; do not invent an origin. Future outcomes, intended recipients and remaining possession time are not action-choice context.

## 6. Context in the learned tasks {#context-in-tasks}

Only the tree-based tasks use context variables. The physical models, ball-drive risk and pass probability, use the tracking state alone.

| Task | Context variables |
|---|---|
| Pass selection | Start type, previous action, match minute and score difference |
| Action selection | Start type, previous action, match minute, score difference and `completed_canonical_drives` |
| Residual risk | Start type, previous action, match minute, score difference and elapsed individual-control time |

Match minute and score difference (carrier-team goals minus opponent goals) are read at the evaluated frame and do not depend on the segmentation. Start type and previous action are one-hot encoded, with buckets for missing and unseen values. The possession-level variables are computed from the observed prefix of the individual possession, never from its eventual duration or outcome. Action selection uses the number of completed canonical carries and not the elapsed time. Carry checkpoints fall on whole seconds, whereas passes are released at arbitrary times, so the elapsed time would reveal which action follows. This follows from the segmentation choices and would be leakage. Residual risk predicts the terminal outcome of the possession and not the choice of action, so it uses the elapsed time.

## 7. An observed sequence {#observed-sequence}

Ivan Vujica’s possession `8_774` in **Melbourne City FC–Macarthur FC** lasts 2.9 s and ends with a completed pass to Luke Brattan. The table below is read from the processed match, including the noncanonical short-tail snapshot. It illustrates both the segmentation and the change in `previous_action`.


In [3]:
import os
from pathlib import Path
import pandas as pd
from PRESS import PRESS

opendata = Path(os.environ.get("SKILLCORNER_OPENDATA", Path.cwd().parent / "opendata"))
processed = Path(os.environ.get(
    "PRESS_PROCESSED_ROOT",
    opendata / "data/processed/press",
))
match_id = "2006229"
match_dir = processed / "matches" / match_id
press = PRESS.from_skillcorner(
    match_dir, actions_path=match_dir / f"{match_id}_actions.parquet"
)
individual_id = "8_774"
individual = press.get_player_possession(individual_id)
checkpoints = press.get_checkpoints(
    individual.team_possession_id,
    include_snapshots=True,
)
checkpoints = checkpoints.loc[checkpoints.possession_id.eq(individual_id)].copy()
checkpoints["elapsed_seconds"] = (
    checkpoints.frame_id - individual.frame_start
) / press.fps
print(checkpoints[
    ["frame_id", "elapsed_seconds", "action", "previous_action", "is_canonical"]
].to_string(index=False))


 frame_id  elapsed_seconds     action previous_action  is_canonical
    41434              0.0 ball_drive  pass_reception          True
    41444              1.0 ball_drive      ball_drive          True
    41454              2.0   snapshot      ball_drive         False
    41463              2.9       pass      ball_drive          True


In [4]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

start = int(individual.frame_start)
end = int(individual.frame_end)
duration = (end - start) / press.fps
segments = press.actions_data.loc[press.actions_data.possession_id.eq(individual_id)]
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.hlines(0, 0, duration, color="#394b59", linewidth=2, zorder=1)
for row in segments.itertuples():
    left = (row.frame_start - start) / press.fps
    right = (row.frame_end - start) / press.fps
    if row.action == "ball_drive":
        ax.hlines(0, left, right, color="#35759b", linewidth=7, zorder=2)
        ax.text((left + right) / 2, 0.32, "Successful carry · 1 s",
                ha="center", fontsize=10, color="#245575")
    elif row.action == "snapshot" and row.snapshot_reason == "short_control_tail":
        ax.hlines(0, left, right, color="#8b969e", linewidth=4,
                  linestyles="dashed", zorder=2)
        ax.text((left + right) / 2, 0.32, f"Control tail · {right-left:.1f} s",
                ha="center", fontsize=10, color="#59636b")
for row in checkpoints.itertuples():
    elapsed = (row.frame_id - start) / press.fps
    if row.is_canonical:
        color = "#b76e2b" if row.action == "pass" else "#245575"
        ax.scatter(elapsed, 0, s=130, c=color, edgecolors="white", zorder=4)
        label = "Pass release" if row.action == "pass" else "Carry start"
    else:
        ax.scatter(elapsed, 0, s=100, facecolors="white", edgecolors="#59636b",
                   linewidths=1.8, zorder=4)
        label = "Extra snapshot"
    ax.annotate(label, (elapsed, 0), xytext=(0, -24), textcoords="offset points",
                ha="center", fontsize=10)
    ax.text(elapsed, -0.55, f"Frame {row.frame_id}", ha="center", fontsize=9,
            color="#66727a")
ax.set(xlim=(-0.2, duration + 0.22), ylim=(-0.85, 0.72), yticks=[],
       xticks=checkpoints.elapsed_seconds,
       xlabel="Seconds since the start of individual control",
       title=f"Ivan Vujica · {duration:.1f} s of control · completed pass")
ax.spines[["left", "right", "top"]].set_visible(False)
ax.spines["bottom"].set_color("#bfc7cd")
ax.legend(handles=[
    Line2D([], [], marker="o", linestyle="none", color="#245575", label="Canonical checkpoint"),
    Line2D([], [], marker="o", linestyle="none", markerfacecolor="white",
           markeredgecolor="#59636b", label="Additional snapshot"),
], loc="upper center", bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig("figures/user_guide_possession_timeline.png", dpi=170)
fig.savefig("figures/user_guide_possession_timeline.svg")
plt.close(fig)


![The line spans Vujica’s observed 2.9 s possession. Filled markers are canonical checkpoints; the open marker at 2 s is an additional snapshot at the start of the short tail.](../../figures/user_guide_possession_timeline.png)

At 0 s and 1 s there are canonical carries. The frame at 2 s is the start of a 0.9 s tail and is an additional snapshot. The pass at 2.9 s supplies the last canonical checkpoint. Previous action is reception at the initial state and becomes carrying after the first complete second.

In [7]:
for frame_id in [41434, 41439, 41444, 41454, 41463]:
    context = press.get_frame_context(frame_id)
    print(frame_id, context["previous_action"], context["completed_canonical_drives"])


41434 pass_reception 0
41439 pass_reception 0
41444 ball_drive 1
41454 ball_drive 2
41463 ball_drive 2


The last column above counts completed canonical carries (0, 0, 1, 2, 2), which is what action selection reads. Residual risk reads the elapsed time instead, which is 0, 0.5, 1, 2 and 2.9 s at the same frames. The [complete usage example](../00_match_workflow/00_match_workflow.ipynb) plots these observed frames and calculates h and xLoss at the three canonical checkpoints and at an intermediate frame.